# MapType in PySpark

This notebook demonstrates how to work with MapType (key-value pairs) in PySpark for handling dictionary-like data structures.

In [ ]:
# Install and setup Java (for Google Colab)
import os

def install_java():
    !apt-get install -y openjdk-8-jdk-headless -qq > /dev/null
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
    !java -version

install_java()

In [ ]:
# Install PySpark
!pip install pyspark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, MapType, IntegerType
from pyspark.sql.functions import col, map_keys, map_values, explode, create_map, lit

spark = SparkSession.builder \
    .appName('MapType in PySpark') \
    .getOrCreate()

print(f"Spark Version: {spark.version}")

## What is MapType?

MapType represents a dictionary or key-value pair structure in PySpark. It's similar to Python dictionaries where each element has a key and a corresponding value.

**Use Cases:**
- Storing dynamic attributes (e.g., product properties)
- Configuration settings
- Metadata with varying keys
- JSON objects with unknown keys

## Creating DataFrame with MapType

### Example 1: Basic MapType with Schema

In [ ]:
# Define schema with MapType
schema = StructType([
    StructField("name", StringType(), True),
    StructField("attributes", MapType(StringType(), StringType()), True)
])

# Sample data - tuples with dictionaries
data = [
    ("John", {"height": "6ft", "eye_color": "blue", "weight": "60"}),
    ("Jane", {"height": "5.5ft", "eye_color": "green"}),
    ("Bob", {"height": "5.8ft", "eye_color": "brown", "weight": "70", "hair_color": "black"}),
    ("Alice", {"eye_color": "hazel", "weight": "55"})
]

# Create DataFrame
df = spark.createDataFrame(data, schema)

print("DataFrame with MapType:")
df.show(truncate=False)

print("\nSchema:")
df.printSchema()

## Accessing Map Values by Key

### Access Specific Keys

In [ ]:
# Access map values using bracket notation
print("Accessing Map Values:")
df2 = df.select(
    col("name"),
    df.attributes["height"].alias("height"),
    df.attributes["eye_color"].alias("eye_color"),
    df.attributes["weight"].alias("weight")
)
df2.show(truncate=False)

### Alternative: Using getItem()

In [ ]:
# Alternative syntax using getItem()
print("Using getItem():")
df.select(
    col("name"),
    col("attributes").getItem("height").alias("height"),
    col("attributes").getItem("eye_color").alias("eye_color")
).show(truncate=False)

## Map Functions

### Get All Keys

In [ ]:
# Extract all keys from map
print("Map Keys:")
df.select(
    col("name"),
    map_keys(col("attributes")).alias("keys")
).show(truncate=False)

### Get All Values

In [ ]:
# Extract all values from map
print("Map Values:")
df.select(
    col("name"),
    map_values(col("attributes")).alias("values")
).show(truncate=False)

### Get Keys and Values Together

In [ ]:
# Show both keys and values
print("Keys and Values:")
df.select(
    col("name"),
    col("attributes"),
    map_keys(col("attributes")).alias("keys"),
    map_values(col("attributes")).alias("values")
).show(truncate=False)

## Exploding Maps

### Convert Map to Rows (Key-Value Pairs)

In [ ]:
# Explode map into separate rows
print("Original DataFrame:")
df.show(truncate=False)

print("\nExploded Map (Key-Value Pairs as Rows):")
df_exploded = df.select(
    col("name"),
    explode(col("attributes")).alias("key", "value")
)
df_exploded.show(truncate=False)

In [ ]:
# Filter exploded data
print("Only Height Attributes:")
df_exploded.filter(col("key") == "height").show(truncate=False)

print("\nOnly Eye Color Attributes:")
df_exploded.filter(col("key") == "eye_color").show(truncate=False)

## Creating Maps Dynamically

### Create Map from Columns

In [ ]:
# Start with flat data
flat_data = [
    ("Product1", "Red", "Large", "Cotton"),
    ("Product2", "Blue", "Medium", "Polyester"),
    ("Product3", "Green", "Small", "Silk")
]

df_flat = spark.createDataFrame(
    flat_data,
    ["product", "color", "size", "material"]
)

print("Flat DataFrame:")
df_flat.show()

# Create map from columns
print("\nDataFrame with Map Created from Columns:")
df_with_map = df_flat.select(
    col("product"),
    create_map(
        lit("color"), col("color"),
        lit("size"), col("size"),
        lit("material"), col("material")
    ).alias("properties")
)
df_with_map.show(truncate=False)
df_with_map.printSchema()

## Filtering Based on Map Contents

In [ ]:
# Filter based on map key existence and value
print("People with height attribute:")
df.filter(col("attributes").getItem("height").isNotNull()).show(truncate=False)

print("\nPeople with blue eyes:")
df.filter(col("attributes")["eye_color"] == "blue").show(truncate=False)

print("\nPeople with weight attribute:")
df.filter(col("attributes").getItem("weight").isNotNull()).show(truncate=False)

## Map with Different Value Types

In [ ]:
# Map with integer values
schema_int = StructType([
    StructField("student", StringType(), True),
    StructField("scores", MapType(StringType(), IntegerType()), True)
])

student_data = [
    ("Alice", {"Math": 95, "Science": 88, "English": 92}),
    ("Bob", {"Math": 78, "Science": 85, "History": 90}),
    ("Charlie", {"Math": 92, "English": 88, "History": 85})
]

df_students = spark.createDataFrame(student_data, schema_int)

print("Student Scores (Map with Integer Values):")
df_students.show(truncate=False)
df_students.printSchema()

In [ ]:
# Access specific subjects
print("Math and Science Scores:")
df_students.select(
    col("student"),
    col("scores")["Math"].alias("math_score"),
    col("scores")["Science"].alias("science_score")
).show()

## Map Size and Operations

In [ ]:
from pyspark.sql.functions import size

# Get number of key-value pairs in map
print("Number of Attributes per Person:")
df.select(
    col("name"),
    col("attributes"),
    size(col("attributes")).alias("num_attributes")
).show(truncate=False)

In [ ]:
# Find people with most attributes
print("Person with Most Attributes:")
df.select(
    col("name"),
    size(col("attributes")).alias("num_attributes")
).orderBy(col("num_attributes").desc()).show()

## Practical Example: Product Catalog

In [ ]:
# Product catalog with dynamic attributes
product_schema = StructType([
    StructField("product_id", StringType(), True),
    StructField("name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("specifications", MapType(StringType(), StringType()), True)
])

products = [
    ("P001", "Laptop", "Electronics", {"brand": "Dell", "ram": "16GB", "storage": "512GB SSD", "screen": "15.6 inch"}),
    ("P002", "Phone", "Electronics", {"brand": "Samsung", "ram": "8GB", "storage": "128GB", "camera": "48MP"}),
    ("P003", "Shirt", "Clothing", {"brand": "Nike", "size": "L", "color": "Blue", "material": "Cotton"}),
    ("P004", "Book", "Media", {"author": "John Doe", "pages": "350", "publisher": "ABC Books"})
]

df_products = spark.createDataFrame(products, product_schema)

print("Product Catalog:")
df_products.show(truncate=False)
df_products.printSchema()

In [ ]:
# Query products by specifications
print("Products with Brand Specification:")
df_products.select(
    col("product_id"),
    col("name"),
    col("category"),
    col("specifications")["brand"].alias("brand")
).filter(col("specifications")["brand"].isNotNull()).show(truncate=False)

print("\nElectronics with RAM:")
df_products.filter(
    (col("category") == "Electronics") & 
    (col("specifications")["ram"].isNotNull())
).select(
    col("name"),
    col("specifications")["brand"].alias("brand"),
    col("specifications")["ram"].alias("ram")
).show(truncate=False)

In [ ]:
# Explode product specifications
print("All Product Specifications (Exploded):")
df_specs_exploded = df_products.select(
    col("product_id"),
    col("name"),
    explode(col("specifications")).alias("spec_name", "spec_value")
)
df_specs_exploded.show(truncate=False)

# Find all products with specific specification
print("\nProducts with 'brand' Specification:")
df_specs_exploded.filter(col("spec_name") == "brand").show(truncate=False)

## Aggregations with Maps

In [ ]:
from pyspark.sql.functions import count, collect_list

# Count products by specification keys
print("Specification Usage Count:")
df_specs_exploded.groupBy("spec_name") \
    .agg(count("*").alias("usage_count")) \
    .orderBy(col("usage_count").desc()) \
    .show()

# Products grouped by category
print("\nProducts per Category:")
df_products.groupBy("category") \
    .agg(
        count("*").alias("product_count"),
        collect_list("name").alias("products")
    ).show(truncate=False)

## Converting Map to JSON

In [ ]:
from pyspark.sql.functions import to_json

# Convert map to JSON string
print("Map as JSON String:")
df.select(
    col("name"),
    to_json(col("attributes")).alias("attributes_json")
).show(truncate=False)

## Handling Missing Keys

In [ ]:
from pyspark.sql.functions import when, coalesce

# Handle missing keys with default values
print("Handling Missing Keys:")
df.select(
    col("name"),
    coalesce(col("attributes")["height"], lit("Unknown")).alias("height"),
    coalesce(col("attributes")["weight"], lit("Unknown")).alias("weight"),
    coalesce(col("attributes")["hair_color"], lit("Unknown")).alias("hair_color")
).show(truncate=False)

## Key Takeaways

### MapType Benefits:
- **Flexible Schema**: Keys don't need to be predefined
- **Dynamic Attributes**: Different rows can have different keys
- **Sparse Data**: Only store attributes that exist
- **JSON-like**: Natural representation of key-value data

### Accessing Map Data:
- **By Key**: `col("map")["key"]` or `col("map").getItem("key")`
- **All Keys**: `map_keys(col("map"))`
- **All Values**: `map_values(col("map"))`
- **Explode**: `explode(col("map"))` for key-value pairs as rows

### Operations:
- **Filter**: Check for key existence or value matching
- **Size**: `size(col("map"))` returns number of entries
- **Create**: Use `create_map()` to build from columns
- **Transform**: Convert to JSON with `to_json()`

### Best Practices:
- Use MapType for dynamic/varying attributes
- Handle missing keys with `coalesce()` or `when()`
- Explode when you need to analyze keys/values separately
- Consider struct if keys are known and fixed

In [ ]:
# Stop Spark Session
spark.stop()